# Evals: AI ka jawab sahi hai, ye kaise jaanchein?
**AI Explained · Episode 18 (full version)** · Jajati AI

Har cell ko upar se neeche chalao (Shift + Enter). Google Colab mein free chalta hai.

Is notebook mein koi asli AI nahi chalta, koi API key nahi chahiye. Ek nakli college help-desk hai, aur uske bot ke do **toy** versions: purana prompt (v1) aur naya, friendly prompt (v2). Asli bot ek LLM hota; hamare toy sirf words milaate hain, taaki eval ka idea saaf dikhe. FAQ aur sawaal hamne teaching ke liye banaye hain.

In [1]:
FAQ = {   # nakli college help-desk
    "library": "9 pm",
    "exam": "2 january",
    "exam form": "15 december",
    "fees": "20 december",
    "late fees": "500 rupaye",
    "result": "5 february",
    "result recheck": "12 february",
    "hostel": "10 pm",
    "canteen": "9:30 pm",
    "placement talk": "friday",
}
print(len(FAQ), "facts")

10 facts


In [2]:
def bot_v1(q):   # purana prompt
    keys = [k for k in FAQ if k in q]
    if not keys:
        return "pata nahi"
    return FAQ[max(keys, key=len)]  # sabse pakka

def bot_v2(q):   # naya prompt: friendly
    for k in FAQ:               # pehla match
        if k in q:
            return "Haan ji! " + FAQ[k]
    return "Maaf kijiye, pata nahi"

In [3]:
# vibe check: sirf ek sawaal
Q = "library kab tak khuli hai"
print("v1:", bot_v1(Q))
print("v2:", bot_v2(Q))

v1: 9 pm
v2: Haan ji! 9 pm


In [4]:
TESTS = [   # (sawaal, sahi jawab)
  ("library kab tak khuli hai", "9 pm"),
  ("exam kab se hai", "2 january"),
  ("exam form ki last date", "15 december"),
  ("fees ki last date", "20 december"),
  ("late fees kitni hai", "500 rupaye"),
  ("result kab aayega", "5 february"),
  ("result recheck ki last date",
   "12 february"),
  ("hostel gate kab band hota hai", "10 pm"),
  ("mess kab tak khula hai", "9:30 pm"),
  ("placement talk kab hai", "friday"),
]
print(len(TESTS), "tests")

10 tests


In [5]:
def contains(reply, gold):  # jawab andar hai?
    return gold in reply.lower()

def score(bot, grader=contains):
    ok = sum(grader(bot(q), g) for q, g in TESTS)
    return f"{ok}/{len(TESTS)}"

print("v1 purana:", score(bot_v1))
print("v2 naya  :", score(bot_v2))

v1 purana: 9/10
v2 naya  : 6/10


In [6]:
# v2 kahan fail hua?
for q, g in TESTS:
    r = bot_v2(q)
    if not contains(r, g):
        print(f"{q:28s} -> {r}")

exam form ki last date       -> Haan ji! 2 january
late fees kitni hai          -> Haan ji! 20 december
result recheck ki last date  -> Haan ji! 5 february
mess kab tak khula hai       -> Maaf kijiye, pata nahi


In [7]:
def exact(reply, gold):   # bilkul same?
    return reply.strip().lower() == gold

print("exact v1:", score(bot_v1, exact))
print("exact v2:", score(bot_v2, exact))

exact v1: 9/10
exact v2: 0/10


In [8]:
def bot_v3(q):   # friendly + sabse pakka
    keys = [k for k in FAQ if k in q]
    if not keys:
        return "Maaf kijiye, pata nahi"
    return "Haan ji! " + FAQ[max(keys, key=len)]

print("v3:", score(bot_v3))
FAQ["mess"] = "9:30 pm"   # fail se seekha
print("v3:", score(bot_v3))

v3: 9/10
v3: 10/10


In [9]:
# user ki shikayat -> naya test
TESTS.append(("library sunday ko khuli hai",
              "band"))
print(len(TESTS), "tests | v3:", score(bot_v3))

11 tests | v3: 10/11


### Asli duniya mein
OpenAI ka *Evaluation best practices* guide: *"Generative AI is variable."* Isliye purane software tests kaafi nahi. Steps: objective tay karo, dataset jodo, metric chuno, evals chalao aur compare karo, aur *"Continuously evaluate."* Guide ek galti ka naam bhi leta hai: *"Vibe-based evals"*, yaani "lag raha hai chal raha hai".

Grader teen tarah ke: **code** (jaise hamara `contains`), **insaan** (*"highest quality but slow and expensive"*), aur **LLM judge** (sasta aur scalable, par pehle insaanon se milaao). Research (Zheng et al. 2023): strong LLM judges insaano se 80% se zyaada baar sahmat hue, utna hi jitna insaan aapas mein; par unmein position aur verbosity jaise bias bhi mile.

Anthropic: evals ko *"task-specific"* rakho, edge cases mat bhoolo, grading ho sake to code se karo. Shuruaat: *"20-50 simple tasks drawn from real failures is a great start."* Bina evals ke: *"wait for complaints, reproduce manually, fix the bug, and hope nothing else regressed."*

In [10]:
ANSWER = ("An eval is a fixed set of test "
  "inputs with known good answers, plus "
  "a grader that scores every output "
  "the same way. I run it on every "
  "prompt or model change and compare "
  "scores, not one answer. Start small, "
  "add real failures, and check an LLM "
  "judge against humans first.")
print(ANSWER)

An eval is a fixed set of test inputs with known good answers, plus a grader that scores every output the same way. I run it on every prompt or model change and compare scores, not one answer. Start small, add real failures, and check an LLM judge against humans first.


In [11]:
# TUMHARI BAARI: apna sawaal aur sahi jawab
TESTS.append(("hostel mein guest kab tak",
              "8 pm"))
print(score(bot_v1), score(bot_v3))

10/12 10/12
